# Lilly v2 — train the reader (OCR on Kaggle)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian only (č ć đ š ž). No Cyrillic.

Set these in the panel on the right:

- **Session options → Accelerator → GPU T4**
- **Session options → Internet → On**

Nothing to upload — clones GitHub, fetches base weights from Hugging Face,
generates ~20k synthetic crops, trains **10 epochs**, ships `lilly-read.zip`.

Real hand-labelled crop **images** are not in git (only labels are). This run is
synthetic-heavy unless you attach a dataset. Every step stops the run if it fails.

In [ ]:
# 1. Stop here unless the machine is actually set up
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

In [ ]:
# 2. Get the Lilly code
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert Path("/kaggle/working/Lilly/training/train_ocr.py").is_file(), "clone produced nothing"
os.chdir("/kaggle/working/Lilly")
print("working in", os.getcwd())

In [ ]:
# 3. Install what we need (~2 min)
# Do NOT pip-install torch from requirements.txt — that file pins CPU wheels for
# the Mac. Kaggle already ships a GPU build; replacing it wastes time and can
# break CUDA.
NEEDED = ["easyocr", "opencv-python-headless", "pillow", "huggingface_hub"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line.startswith("--"):
        continue
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 3b. Prove the GPU can backprop before an hour is spent generating images
x = torch.randn(256, 256, device="cuda", requires_grad=True)
y = (x @ torch.randn(256, 256, device="cuda")).sum()
y.backward()
print(f"GPU backprop ok on {torch.cuda.get_device_name(0)}")
torch.cuda.empty_cache()

In [ ]:
# 4. Base reader weights from Hugging Face (~easyocr latin_g2 starting point)
run("python3", "scripts/fetch_models.py")
base = Path("models/lilly/read/latin_g2.pth")
assert base.is_file() and base.stat().st_size > 1_000_000, "fetch_models missing read weights"

In [ ]:
# 5. Synthetic Bosnian crops — keep PNGs off /kaggle/working (Output bloat)
scratch = Path("/kaggle/temp/ocr-synthetic" if Path("/kaggle/temp").is_dir()
             else "/tmp/lilly-ocr-synthetic")
scratch.mkdir(parents=True, exist_ok=True)
syn = Path("data/ocr/synthetic")
if syn.is_symlink():
    syn.unlink()
elif syn.is_dir():
    subprocess.run(["rm", "-rf", str(syn)], check=True)
syn.symlink_to(scratch)

# Fonts: repo has data/fonts/README only. generate_ocr_data.py falls back to
# /usr/share/fonts on Linux and exits with a clear error if none have č/đ.
run("python3", "data/scripts/generate_ocr_data.py", "--count", "20000", "--build-splits")
train_n = sum(1 for _ in open("data/ocr/train/gt.txt", encoding="utf-8"))
valid_n = sum(1 for _ in open("data/ocr/valid/gt.txt", encoding="utf-8"))
assert train_n > 5000, f"only {train_n} train crops"
assert valid_n > 100, f"only {valid_n} valid crops"
print(f"synthetic splits: train {train_n:,}, valid {valid_n:,}")

In [ ]:
# 5b. Real hand-labelled crops — only if the PNGs came with the clone
for labels in (Path("data/ocr/crops2/labels-human.tsv"),
               Path("data/ocr/crops/labels-human.tsv")):
    if not labels.is_file():
        continue
    first_line = labels.read_text(encoding="utf-8").splitlines()[0]
    img_name = first_line.split("\t")[0]
    img_path = labels.parent / img_name
    if img_path.is_file():
        run("python3", "training/prepare_ocr_data.py", "--labels", str(labels))
        print(f"merged real crops from {labels}")
        break
else:
    print("no real crop images in clone — synthetic only (expected on Kaggle)")

In [ ]:
# 6. THE REAL TRAINING (~4-8 h on T4, 10 epochs)
# --keep-trained writes weights even if the install gate refuses a regression.
# train_ocr may exit 1 when the gate refuses — that is NOT a failed train.
TRAINED = Path("models/lilly/read-trained.pth")
cmd = ["python3", "training/train_ocr.py", "--epochs", "10", "--batch-size", "16",
       "--keep-trained", str(TRAINED)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False)
assert TRAINED.is_file() and TRAINED.stat().st_size > 100_000, (
    f"training produced nothing (exit {proc.returncode})")
if proc.returncode != 0:
    print(f"train_ocr exited {proc.returncode} — install gate may have refused; "
          f"keep-trained saved ({TRAINED.stat().st_size:,} bytes)")

# Zip immediately — do not lose a finished train to a later packaging step.
run("zip", "-j", "/kaggle/working/lilly-read-trained.zip", str(TRAINED))
print("saved /kaggle/working/lilly-read-trained.zip before app packaging")

In [ ]:
# 7. Package what the app loads (lilly.pth + yaml + py)
import shutil
app_weights = Path("models/lilly/read/lilly.pth")
if not app_weights.exists() or app_weights.stat().st_mtime < TRAINED.stat().st_mtime:
    shutil.copy(TRAINED, app_weights)
    print("copied read-trained.pth -> lilly.pth for the app")

for needed in ("lilly.yaml", "lilly.py"):
    assert Path("models/lilly/read", needed).is_file(), f"missing {needed}"

run("zip", "-qr", "/kaggle/working/lilly-read.zip",
    "models/lilly/read/lilly.pth", "models/lilly/read/lilly.yaml",
    "models/lilly/read/lilly.py")
size = Path("/kaggle/working/lilly-read.zip").stat().st_size
assert size > 100_000, f"zip too small: {size}"
print(f"lilly-read.zip — {size / 1048576:.1f} MB")
print("lilly-read-trained.zip is in Output too if packaging above failed")

**Done.** Download `lilly-read.zip` from **Output** and unzip into `models/lilly/read/`.
Keep `latin_g2.pth` and any previous `lilly.pth` as backup.

If word/diacritic scores in the log did not rise, do not install — train longer or
add real labelled crops before another run.